# Cross-Sectional Momentum on S&P 500

**Research question:** Do past winners continue to outperform past losers in the S&P 500?

**Hypothesis:** Stocks with the highest returns over the past 12 months will outperform stocks with the lowest returns over the following month.

In [3]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings('ignore')

## 1. Data Collection
Download daily adjusted close prices for all S&P 500 constituents.

In [4]:
response = requests.get("https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv")
data_frame = pd.read_csv(pd.io.common.StringIO(response.text))

list_sp500 = data_frame["Symbol"].tolist()
print(list_sp500[:10])

['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


## 2. Download Historical Prices
We download daily adjusted close prices for all S&P 500 constituents since 2010. Adjusted close accounts for dividends and stock splits, ensuring consistent return calculations.

In [5]:
# Fix Yahoo Finance ticker format
list_sp500 = [t.replace('.', '-') for t in list_sp500]

data_candles = yf.download(list_sp500, "2010-01-01", auto_adjust=True)

print(data_candles.shape)
print(data_candles.head())

[*********************100%***********************]  503 of 503 completed


(4233, 2515)
Price           Close                                                      \
Ticker              A      AAPL ABBV ABNB        ABT      ACGL        ACN   
Date                                                                        
2010-01-04  19.743517  6.400959  NaN  NaN  17.968189  7.601905  30.610767   
2010-01-05  19.529057  6.412027  NaN  NaN  17.823025  7.576549  30.799957   
2010-01-06  19.459661  6.310034  NaN  NaN  17.922003  7.543795  31.127380   
2010-01-07  19.434433  6.298369  NaN  NaN  18.070467  7.499420  31.098267   
2010-01-08  19.428120  6.340243  NaN  NaN  18.162851  7.484628  30.974575   

Price                                        ...   Volume                    \
Ticker           ADBE        ADI        ADM  ...       WY     WYNN      XEL   
Date                                         ...                              
2010-01-04  37.090000  21.703714  20.029282  ...  1832400  4741400  2670400   
2010-01-05  37.700001  21.669455  20.137482  ...  1724

In [6]:
price = data_candles["Close"]
print(price.head())

Ticker              A      AAPL  ABBV  ABNB        ABT      ACGL        ACN  \
Date                                                                          
2010-01-04  19.743517  6.400959   NaN   NaN  17.968189  7.601905  30.610767   
2010-01-05  19.529057  6.412027   NaN   NaN  17.823025  7.576549  30.799957   
2010-01-06  19.459661  6.310034   NaN   NaN  17.922003  7.543795  31.127380   
2010-01-07  19.434433  6.298369   NaN   NaN  18.070467  7.499420  31.098267   
2010-01-08  19.428120  6.340243   NaN   NaN  18.162851  7.484628  30.974575   

Ticker           ADBE        ADI        ADM  ...        WY       WYNN  \
Date                                         ...                        
2010-01-04  37.090000  21.703714  20.029282  ...  9.157350  40.493717   
2010-01-05  37.700001  21.669455  20.137482  ...  9.351745  42.956516   
2010-01-06  37.619999  21.628340  20.086561  ...  9.248343  42.393063   
2010-01-07  36.889999  21.457005  19.876528  ...  9.206985  43.298412   
2010-01-

## 3. Compute Monthly Returns
We resample daily prices to monthly frequency and compute log returns. Log returns are additive over time, which simplifies the calculation of cumulative multi-month returns needed for the momentum signal.

In [7]:
monthly_price = price.resample('ME').last()
monthly_log = np.log(monthly_price)
monthly_yield = monthly_log.diff()
print(monthly_yield.head())

Ticker             A      AAPL  ABBV  ABNB       ABT      ACGL       ACN  \
Date                                                                       
2010-01-31       NaN       NaN   NaN   NaN       NaN       NaN       NaN   
2010-02-28  0.115442  0.063347   NaN   NaN  0.024997  0.033538 -0.025199   
2010-03-31  0.089049  0.138430   NaN   NaN -0.029920  0.030223  0.048349   
2010-04-30  0.052950  0.105280   NaN   NaN -0.020889 -0.008826  0.048234   
2010-05-31 -0.113793 -0.016256   NaN   NaN -0.072966 -0.027634 -0.151100   

Ticker          ADBE       ADI       ADM  ...        WY      WYNN       XEL  \
Date                                      ...                                 
2010-01-31       NaN       NaN       NaN  ...       NaN       NaN       NaN   
2010-02-28  0.070231  0.081183 -0.015513  ...  0.012453  0.026945  0.001442   
2010-03-31  0.020566 -0.007693 -0.015792  ...  0.113815  0.176352  0.030028   
2010-04-30 -0.051338  0.037792 -0.033782  ...  0.090730  0.151567  0.025

## 4. Momentum Signal
For each stock, we compute the cumulative log return over the past 12 months. This is our momentum score: stocks with high past returns are "winners", stocks with low past returns are "losers".

In [8]:
annual_yield = monthly_yield.rolling(12).sum()
print(annual_yield.tail())

Ticker             A      AAPL      ABBV      ABNB       ABT      ACGL  \
Date                                                                     
2026-06-30  0.128397  0.347792  0.336357  0.078169 -0.384508  0.063920   
2026-07-31  0.194645  0.401541  0.313912  0.134814 -0.154333  0.155411   
2026-08-31  0.208504  0.314775  0.228358  0.339085 -0.161191  0.069015   
2026-09-30  0.300521  0.272078  0.152577  0.279972 -0.281140  0.017806   
2026-10-31  0.185450  0.246367  0.331414  0.261683 -0.127337  0.208884   

Ticker           ACN      ADBE       ADI       ADM  ...        WY      WYNN  \
Date                                                ...                       
2026-06-30 -0.862818 -0.635007  0.525344  0.400865  ... -0.035783  0.045022   
2026-07-31 -0.455720 -0.356567  0.505358  0.411562  ...  0.033970 -0.084215   
2026-08-31 -0.294409 -0.197439  0.378678  0.289418  ... -0.074988 -0.318965   
2026-09-30 -0.275865 -0.385371  0.491524  0.313044  ... -0.227574 -0.469941   
2026-10

## 5. Portfolio Construction
Each month, we rank all stocks by their 12-month momentum score and sort them into deciles (10 groups). Decile 1 contains the biggest losers, decile 10 the biggest winners.

In [20]:
ranked_yield = annual_yield.rank(axis=1, pct=True)
print(ranked_yield.tail())

Ticker             A      AAPL      ABBV      ABNB       ABT      ACGL  \
Date                                                                     
2026-06-30  0.507014  0.765531  0.753507  0.442886  0.090180  0.424850   
2026-07-31  0.595190  0.827655  0.733467  0.509018  0.170341  0.547094   
2026-08-31  0.653307  0.777555  0.685371  0.811623  0.154309  0.452906   
2026-09-30  0.803607  0.769539  0.651303  0.785571  0.136273  0.456914   
2026-10-31  0.594000  0.690000  0.804000  0.712000  0.152000  0.630000   

Ticker           ACN      ADBE       ADI       ADM  ...        WY      WYNN  \
Date                                                ...                       
2026-06-30  0.016032  0.038076  0.893788  0.819639  ...  0.286573  0.400802   
2026-07-31  0.042084  0.072144  0.885772  0.837675  ...  0.360721  0.216433   
2026-08-31  0.076152  0.132265  0.843687  0.761523  ...  0.226453  0.056112   
2026-09-30  0.140281  0.080160  0.911824  0.819639  ...  0.174349  0.058116   
2026-10

In [22]:
deciles = np.ceil(ranked_yield * 10)
print(deciles.tail())

Ticker        A  AAPL  ABBV  ABNB  ABT  ACGL  ACN  ADBE   ADI  ADM  ...   WY  \
Date                                                                ...        
2026-06-30  6.0   8.0   8.0   5.0  1.0   5.0  1.0   1.0   9.0  9.0  ...  3.0   
2026-07-31  6.0   9.0   8.0   6.0  2.0   6.0  1.0   1.0   9.0  9.0  ...  4.0   
2026-08-31  7.0   8.0   7.0   9.0  2.0   5.0  1.0   2.0   9.0  8.0  ...  3.0   
2026-09-30  9.0   8.0   7.0   8.0  2.0   5.0  2.0   1.0  10.0  9.0  ...  2.0   
2026-10-31  6.0   7.0   9.0   8.0  2.0   7.0  4.0   1.0  10.0  8.0  ...  2.0   

Ticker      WYNN  XEL   XOM  XYL  XYZ  YUM  ZBH  ZBRA  ZTS  
Date                                                        
2026-06-30   5.0  7.0   7.0  3.0  5.0  5.0  3.0   2.0  1.0  
2026-07-31   3.0  5.0   8.0  2.0  4.0  5.0  4.0   2.0  1.0  
2026-08-31   1.0  5.0   9.0  2.0  4.0  5.0  3.0   6.0  1.0  
2026-09-30   1.0  3.0   9.0  1.0  5.0  4.0  3.0   8.0  1.0  
2026-10-31   1.0  4.0  10.0  1.0  3.0  6.0  3.0   8.0  1.0  

[5 rows x 5